# 🌏 NZ Earthquake Declustering — Method 2: HDBSCAN
### Hierarchical Density-Based Spatial Clustering of Applications with Noise

**Core idea:** Group events that are *dense* in feature space — no need to pre-specify number of clusters, no distance threshold required.

**Why HDBSCAN over plain DBSCAN for seismicity?**
- Handles **variable-density** clusters (NZ has both subduction + crustal seismicity)
- Automatically marks sparse/isolated events as noise (→ excellent background candidates)
- Produces **soft probabilities** per event (membership strength)
- Works directly on your existing 22 features (T1–T10, R1–R10, Mn, bval)

---
### Pipeline
```
Load → Clean → 22 Features → UMAP (dim reduction) → HDBSCAN Clustering
     → Classify (crisis/non-crisis) → Probabilities → Plots → Save
```
---
> **Noise events** (HDBSCAN label = -1) → classified as **non-crisis (background)**  
> **Cluster events** → classified as **crisis** based on cluster density & inter-event distances

## 📦 Step 0 — Install & Import Libraries

In [ ]:
# Uncomment to install:
# !pip install hdbscan umap-learn scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    import hdbscan
    print('✅ hdbscan loaded')
except ImportError:
    raise ImportError('Run: pip install hdbscan')

try:
    import umap
    print('✅ umap loaded')
except ImportError:
    print('⚠️  umap not found — UMAP visualisation will be skipped')
    umap = None

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
def out(f): return str(OUTPUT_DIR / f)

print('✅ All imports done!')

---
## 📂 Step 1 — Load & Clean Data
> 🔧 Change `YOUR_CSV` to your actual file path.

In [ ]:
YOUR_CSV = 'som_feature_nz_real_catalog.csv'

df = pd.read_csv(YOUR_CSV, low_memory=False)
print(f'Raw shape: {df.shape}')

t_cols = [f'T{i}' for i in range(1,11)]
r_cols = [f'R{i}' for i in range(1,11)]
required = t_cols + r_cols + ['Mn','bval']

df_clean = df.dropna(subset=required).copy()
if 'i+' in df_clean.columns:
    df_clean = df_clean[df_clean['i+'] > 0].copy()
df_clean = df_clean.reset_index(drop=True)
print(f'Clean shape: {df_clean.shape}')

---
## 🧮 Step 2 — Build & Scale Feature Matrix

We use the same 22 features as KDM for direct comparison:
- **T1–T10**: temporal distances to 10 nearest neighbours
- **R1–R10**: spatial distances to 10 nearest neighbours
- **Mn**: magnitude ratio (STA/LTA style)
- **bval**: local b-value

HDBSCAN is sensitive to scale — we use **StandardScaler** (mean=0, std=1)
which preserves relative distances better than MinMax for density-based methods.

In [ ]:
feature_cols = t_cols + r_cols + ['Mn','bval']

X = df_clean[feature_cols].values.astype(np.float32)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# StandardScaler is preferred for density-based clustering
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature matrix : {X_scaled.shape}')
print(f'Memory         : {X_scaled.nbytes/1e6:.1f} MB')

# Quick check: feature variance
pd.DataFrame(X_scaled, columns=feature_cols).describe().round(3)

---
## 🗺️ Step 3 — UMAP Dimensionality Reduction (Optional but Recommended)

UMAP compresses 22 features → 2D while preserving **non-linear structure**.
This makes HDBSCAN faster AND reveals the natural cluster topology of your catalogue.

| Parameter | Value | Why |
|---|---|---|
| `n_neighbors` | 30 | Balance between local and global structure |
| `min_dist` | 0.0 | Tighter clusters (better for HDBSCAN input) |
| `n_components` | 2 | 2D for visualisation; use 10 for clustering only |

> ⏱️ UMAP on 380k events: ~5–10 min on workstation. Skip and use PCA if too slow.

In [ ]:
# ── UMAP parameters ──────────────────────────────────────
USE_UMAP      = True   # set False to use PCA instead
UMAP_NEIGHBORS = 30
UMAP_MIN_DIST  = 0.0
UMAP_COMPONENTS = 2    # 2 for vis; increase to 10 for clustering quality
# ─────────────────────────────────────────────────────────

if USE_UMAP and umap is not None:
    print('Running UMAP...')
    reducer = umap.UMAP(
        n_neighbors  = UMAP_NEIGHBORS,
        min_dist     = UMAP_MIN_DIST,
        n_components = UMAP_COMPONENTS,
        metric       = 'euclidean',
        random_state = 42,
        verbose      = True
    )
    X_embed = reducer.fit_transform(X_scaled)
    print(f'✅ UMAP done. Embedding shape: {X_embed.shape}')
else:
    print('Using PCA instead of UMAP...')
    pca = PCA(n_components=10, random_state=42)
    X_embed = pca.fit_transform(X_scaled)
    print(f'✅ PCA done. Variance explained: {pca.explained_variance_ratio_.sum():.2%}')

# Plot the 2D embedding
if X_embed.shape[1] == 2:
    fig, ax = plt.subplots(figsize=(10,8))
    ax.scatter(X_embed[:,0], X_embed[:,1], s=0.3, alpha=0.3, c='steelblue')
    ax.set_title('UMAP Embedding of NZ Earthquake Features (pre-clustering)', fontsize=13)
    ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
    plt.tight_layout()
    plt.savefig(out('HDBSCAN_umap_embedding.png'), dpi=150)
    plt.show()

---
## 🔍 Step 4 — Run HDBSCAN Clustering

**Key hyperparameters:**

| Parameter | Meaning | NZ recommendation |
|---|---|---|
| `min_cluster_size` | Minimum events to form a cluster | 50–200 |
| `min_samples` | Controls noise sensitivity (higher = more noise) | 10–50 |
| `cluster_selection_epsilon` | Merge nearby clusters below this distance | 0.0–0.5 |
| `prediction_data=True` | Enables soft probability scores | Always True |

**How HDBSCAN labels map to seismicity:**
```
Label = -1  →  NOISE  →  Non-crisis (background, isolated events)
Label ≥  0  →  CLUSTER →  Crisis (aftershock sequences / swarms)
```

In [ ]:
# ── HDBSCAN hyperparameters ───────────────────────────────
MIN_CLUSTER_SIZE = 100   # min events to be a cluster (tune this first)
MIN_SAMPLES      = 20    # controls noise boundary (higher = more noise)
EPSILON          = 0.0   # merge threshold (0 = no merging)
# ─────────────────────────────────────────────────────────

print(f'Running HDBSCAN on {X_embed.shape[0]:,} events...')
print(f'  min_cluster_size = {MIN_CLUSTER_SIZE}')
print(f'  min_samples      = {MIN_SAMPLES}')

clusterer = hdbscan.HDBSCAN(
    min_cluster_size          = MIN_CLUSTER_SIZE,
    min_samples               = MIN_SAMPLES,
    cluster_selection_epsilon = EPSILON,
    prediction_data           = True,   # needed for soft membership
    core_dist_n_jobs          = -1      # use all CPU cores
)
clusterer.fit(X_embed)

labels      = clusterer.labels_
probs       = clusterer.probabilities_   # membership strength 0–1
n_clusters  = len(set(labels)) - (1 if -1 in labels else 0)
n_noise     = (labels == -1).sum()

print(f'\n── HDBSCAN Results ─────────────────────────────')
print(f'  Clusters found : {n_clusters}')
print(f'  Noise events   : {n_noise:,}  ({100*n_noise/len(labels):.1f}%)')
print(f'  Clustered      : {len(labels)-n_noise:,}  ({100*(len(labels)-n_noise)/len(labels):.1f}%)')
print(f'────────────────────────────────────────────────')

# Cluster size distribution
unique, counts = np.unique(labels[labels >= 0], return_counts=True)
print(f'\nCluster sizes — min: {counts.min():,}  max: {counts.max():,}  mean: {counts.mean():.0f}')

---
## 🏷️ Step 5 — Classify Events as Crisis / Non-Crisis

**Classification logic:**
```
Noise (label=-1)              → non_crisis  (isolated background events)
Cluster member (label≥0)
  + membership probability    → P_crisis = probability score
  + cluster density check     → high-density cluster = crisis
```
We also compute a **confidence score** equivalent to KDM's formulation.

In [ ]:
def classify_hdbscan(df_clean, labels, probs, clusterer):
    df = df_clean.copy()
    df['hdbscan_label']  = labels
    df['membership_prob'] = probs

    # P_crisis: clustered events get their membership prob; noise gets ~0
    df['P_crisis']     = np.where(labels == -1, 0.05, probs)
    df['P_non_crisis'] = 1.0 - df['P_crisis']

    # Label: noise → non_crisis; clusters → crisis
    df['label'] = np.where(labels == -1, 'non_crisis', 'crisis')

    # Confidence: how decisive is the classification
    df['confidence'] = np.abs(0.5 - df['P_crisis'].clip(0,1)) / 0.5

    n_c  = (df['label'] == 'crisis').sum()
    n_nc = (df['label'] == 'non_crisis').sum()
    print(f'\n── Classification Summary ──────────────────────')
    print(f'  Total events  : {len(df):,}')
    print(f'  Crisis        : {n_c:,}  ({100*n_c/len(df):.1f}%)')
    print(f'  Non-crisis    : {n_nc:,}  ({100*n_nc/len(df):.1f}%)')
    print(f'  High conf >0.8: {(df["confidence"]>0.8).sum():,}')
    print(f'────────────────────────────────────────────────')
    return df

df_labelled = classify_hdbscan(df_clean, labels, probs, clusterer)
df_labelled[['latitude','longitude','magnitude','hdbscan_label',
             'P_crisis','label','confidence']].head(10)

---
## ⚙️ Step 6 — Hyperparameter Tuning
Test multiple `min_cluster_size` values and compare crisis/non-crisis ratios.
Stable ratio across parameters = robust result.

In [ ]:
# Quick sensitivity test
test_sizes = [50, 100, 200, 500]
results_tune = []

for mcs in test_sizes:
    c_test = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=MIN_SAMPLES,
                              prediction_data=False, core_dist_n_jobs=-1)
    c_test.fit(X_embed)
    lbl = c_test.labels_
    n_cl  = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_no  = (lbl == -1).sum()
    pct_crisis = 100*(len(lbl)-n_no)/len(lbl)
    results_tune.append({'min_cluster_size': mcs, 'n_clusters': n_cl,
                          'n_noise': n_no, 'pct_crisis': round(pct_crisis,1)})
    print(f'  mcs={mcs:4d} → {n_cl} clusters | '
          f'{n_no:,} noise ({100*n_no/len(lbl):.1f}%) | '
          f'crisis {pct_crisis:.1f}%')

tune_df = pd.DataFrame(results_tune)

fig, axes = plt.subplots(1,3, figsize=(15,4))
axes[0].plot(tune_df['min_cluster_size'], tune_df['n_clusters'],  'bo-')
axes[0].set_title('N Clusters vs min_cluster_size'); axes[0].set_xlabel('min_cluster_size')
axes[1].plot(tune_df['min_cluster_size'], tune_df['n_noise'],     'ro-')
axes[1].set_title('Noise Events vs min_cluster_size'); axes[1].set_xlabel('min_cluster_size')
axes[2].plot(tune_df['min_cluster_size'], tune_df['pct_crisis'],  'go-')
axes[2].set_title('% Crisis vs min_cluster_size'); axes[2].set_xlabel('min_cluster_size')
plt.suptitle('HDBSCAN Sensitivity to min_cluster_size', fontsize=13)
plt.tight_layout()
plt.savefig(out('HDBSCAN_tuning.png'), dpi=150)
plt.show()

---
## 📈 Step 7 — Visualisations
### Plot A — UMAP Coloured by HDBSCAN Cluster

In [ ]:
if X_embed.shape[1] == 2:
    fig, axes = plt.subplots(1,2, figsize=(16,7))

    # Panel 1: cluster colours
    unique_lbls = sorted(set(labels))
    cmap = plt.cm.get_cmap('tab20', max(len(unique_lbls),1))
    for lbl in unique_lbls:
        mask  = labels == lbl
        color = 'lightgrey' if lbl == -1 else cmap(lbl % 20)
        name  = 'Noise/Background' if lbl == -1 else f'Cluster {lbl}'
        alpha = 0.2 if lbl == -1 else 0.6
        axes[0].scatter(X_embed[mask,0], X_embed[mask,1],
                        c=[color], s=0.5, alpha=alpha, label=name)
    axes[0].set_title(f'HDBSCAN Clusters ({n_clusters} clusters)', fontsize=12)
    axes[0].set_xlabel('UMAP-1'); axes[0].set_ylabel('UMAP-2')

    # Panel 2: crisis probability
    sc = axes[1].scatter(X_embed[:,0], X_embed[:,1],
                         c=df_labelled['P_crisis'], cmap='RdYlGn_r',
                         s=0.5, alpha=0.5, vmin=0, vmax=1)
    plt.colorbar(sc, ax=axes[1], label='P(crisis)')
    axes[1].set_title('P(crisis) in UMAP Space', fontsize=12)
    axes[1].set_xlabel('UMAP-1'); axes[1].set_ylabel('UMAP-2')

    plt.suptitle('HDBSCAN — NZ Earthquake Clustering', fontsize=14)
    plt.tight_layout()
    plt.savefig(out('HDBSCAN_umap_clusters.png'), dpi=150)
    plt.show()

### Plot B — Spatial Map (NZ Geography)

In [ ]:
crisis     = df_labelled[df_labelled['label']=='crisis']
non_crisis = df_labelled[df_labelled['label']=='non_crisis']

fig, axes = plt.subplots(1,2, figsize=(16,7))
axes[0].scatter(non_crisis['longitude'], non_crisis['latitude'],
                c='steelblue', s=0.5, alpha=0.3, label=f'Non-crisis ({len(non_crisis):,})')
axes[0].scatter(crisis['longitude'], crisis['latitude'],
                c='crimson', s=0.5, alpha=0.3, label=f'Crisis ({len(crisis):,})')
axes[0].set_title('HDBSCAN — Crisis vs Non-Crisis (NZ)', fontsize=12)
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].legend(markerscale=8)

sc = axes[1].scatter(df_labelled['longitude'], df_labelled['latitude'],
                     c=df_labelled['P_crisis'], cmap='RdYlGn_r',
                     s=0.5, alpha=0.4, vmin=0, vmax=1)
plt.colorbar(sc, ax=axes[1], label='P(crisis)')
axes[1].set_title('P(crisis) Gradient Map', fontsize=12)
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(out('HDBSCAN_spatial_map.png'), dpi=150)
plt.show()

### Plot C — Cumulative Curves & Magnitude Distribution

In [ ]:
time_col = 'time' if 'time' in df_labelled.columns else 'Year'
df_sorted = df_labelled.sort_values(time_col)
c_s  = df_sorted[df_sorted['label']=='crisis']
nc_s = df_sorted[df_sorted['label']=='non_crisis']
N    = len(df_sorted)

fig, axes = plt.subplots(1,2, figsize=(16,5))

# Cumulative
axes[0].plot(df_sorted[time_col], np.arange(N)/N,
             'k--', lw=1, alpha=0.5, label='Full catalogue')
axes[0].plot(c_s[time_col],  np.arange(len(c_s))/N,
             'r:', lw=1.5, label=f'Crisis ({len(c_s):,})')
axes[0].plot(nc_s[time_col], np.arange(len(nc_s))/N,
             'b-', lw=1.5, label=f'Non-crisis ({len(nc_s):,})')
axes[0].set_title('Cumulative Curves — HDBSCAN', fontsize=12)
axes[0].set_xlabel('Time (decimal years)'); axes[0].set_ylabel('Proportion')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Magnitude
bins = np.arange(0, df_labelled['magnitude'].max()+0.5, 0.5)
axes[1].hist(c_s['magnitude'],  bins=bins, density=True, alpha=0.6,
             color='orange',    label='Crisis')
axes[1].hist(nc_s['magnitude'], bins=bins, density=True, alpha=0.6,
             color='steelblue', label='Non-crisis')
axes[1].set_title('Magnitude Distribution — HDBSCAN', fontsize=12)
axes[1].set_xlabel('Magnitude'); axes[1].set_ylabel('Density')
axes[1].legend()
plt.tight_layout()
plt.savefig(out('HDBSCAN_cumulative_magnitude.png'), dpi=150)
plt.show()

---
## 💾 Step 8 — Save Results

In [ ]:
save_cols = [c for c in ['event','DateTime','latitude','longitude','depth',
             'magnitude','time','hdbscan_label','P_crisis','P_non_crisis',
             'label','confidence'] if c in df_labelled.columns]
df_labelled[save_cols].to_csv(out('NZ_HDBSCAN_declustered.csv'), index=False)

n_c  = (df_labelled['label']=='crisis').sum()
n_nc = (df_labelled['label']=='non_crisis').sum()
print('✅ Saved → NZ_HDBSCAN_declustered.csv')
print(f'\n══ HDBSCAN Final Summary ═══════════════════')
print(f'  Total    : {len(df_labelled):,}')
print(f'  Crisis   : {n_c:,}  ({100*n_c/len(df_labelled):.1f}%)')
print(f'  Non-crisis: {n_nc:,}  ({100*n_nc/len(df_labelled):.1f}%)')
print(f'  Clusters : {n_clusters}')
print(f'═════════════════════════════════════════════')